# SPLINK Record Linkage Package
This is a notebook authored by Aniyah that attempts to use the Python SPLINK package to perform record matching across Ablemarle County's census data for 1870 and 1880


For future push:
git push -u origin aniyah


## What does Splink do?
Splink matches records by predicting which rows link together. It clusters the links to produce a unique person ID. 

Splink performs best with input data containing multiple columns that are not highly correlated.

Link: https://moj-analytical-services.github.io/splink/getting_started.html

## Set-Up Code

In [1]:
# loading important packages
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
import plotly.express as px
import splink.comparison_library as cl
from splink import DuckDBAPI, Linker, SettingsCreator, block_on, splink_datasets

In [2]:
# loading in the data
mentions_new = pd.read_csv('mentions_new.csv')
mentions_new

,mention_id,source,source_year,county,confidence,full_name,first_name,middle_name,last_name,maiden_name,...,nysiis_last_name,soundex_last_name,norm_race,norm_occupation,location_id,head,household_id,family_id,narrative,created
0,ALB-VR-1715-8702.1,ALB_VR_1715,1858,ALB,0.85,Lucy,Lucy,NaN,Lucy,NaN,...,LACY,L200,NaN,NaN,NaN,NaN,NaN,NaN,Lucy (F in Alb).,2026-06-10T18:25:01.928669+00:00
1,ALB-VR-1715-8703.1,ALB_VR_1715,1858,ALB,0.85,Cynthia,Cynthia,NaN,Cynthia,NaN,...,CYNTA,C530,NaN,NaN,NaN,NaN,NaN,NaN,Cynthia (F in Alb).,2026-06-10T18:25:01.928669+00:00
2,ALB-VR-1715-8704.1,ALB_VR_1715,1858,ALB,0.85,Mary,Mary,NaN,Mary,NaN,...,MARY,M600,NaN,NaN,NaN,NaN,NaN,NaN,Mary (F in Alb).,2026-06-10T18:25:01.928669+00:00
3,ALB-VR-1715-8705.1,ALB_VR_1715,1858,ALB,0.85,Mildred,Mildred,NaN,Mildred,NaN,...,MALDRAD,M436,NaN,NaN,NaN,NaN,NaN,NaN,Mildred (F in Alb).,2026-06-10T18:25:01.928669+00:00
4,ALB-VR-1715-8706.1,ALB_VR_1715,1858,ALB,0.85,Lucinda,Lucinda,NaN,Lucinda,NaN,...,LACAND,L253,NaN,NaN,NaN,NaN,NaN,NaN,Lucinda (F in Alb).,2026-06-10T18:25:01.928669+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118033,ALB-CN-1880-28630,ALB_CN_1880,1880,ALB,0.90,Sarah Bryan,Sarah,NaN,Bryan,NaN,...,BRYAN,B650,W,DOMESTIC,NaN,True,NaN,FC1880-4969,SARAH BRYAN is a White female born in 1820. Sh...,2026-06-10T18:20:44.716359+00:00
118034,ALB-CN-1880-28631,ALB_CN_1880,1880,ALB,0.90,Martha Bryan,Martha,NaN,Bryan,NaN,...,BRYAN,B650,W,DOMESTIC,NaN,False,NaN,FC1880-4969,MARTHA BRYAN is a White female born in 1853 wi...,2026-06-10T18:20:44.716359+00:00
118035,ALB-CN-1880-28632,ALB_CN_1880,1880,ALB,0.90,Catharine Bryan,Catharine,NaN,Bryan,NaN,...,BRYAN,B650,W,DOMESTIC,NaN,False,NaN,FC1880-4969,CATHARINE BRYAN is a White female born in 1855...,2026-06-10T18:20:44.716359+00:00
118036,ALB-SS-1860-4522,ALB_SS-1860,1860,ALB,0.82,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,B,NaN,NaN,False,NaN,NaN,Unnamed (M / B born 1859 in Alb).,2026-06-10T18:41:59.981257+00:00


In [3]:
mentions_new['source'].value_counts()

source
ALB_CN_1880       40302
ALB_VR_1715       20426
ALB_CN_1870       18985
ALB_SS-1860       16945
ALB_SS-1850       15335
ALB_CH_1851        2578
ALB_FindAGrave     1630
ALB_FBR            1447
ALB_FL-1865         390
Name: count, dtype: int64

In [4]:
# need to filter the data to only the census data from 1870 and 1880
mentions = mentions_new[
    (mentions_new['source'] == 'ALB_CN_1870')|
     (mentions_new['source'] == 'ALB_CN_1880')  
    ]
mentions

,mention_id,source,source_year,county,confidence,full_name,first_name,middle_name,last_name,maiden_name,...,nysiis_last_name,soundex_last_name,norm_race,norm_occupation,location_id,head,household_id,family_id,narrative,created
39,ALB-CN-1880-27906,ALB_CN_1880,1880,ALB,0.9,Anna M Stickley,Anna,M,Stickley,NaN,...,STACLAY,S324,W,DOMESTIC,NaN,False,NaN,FC1880-4835,Anna M Stickley (F / W born 1876 in Alb). Pare...,2026-06-10T18:20:40.886432+00:00
40,ALB-CN-1880-1377,ALB_CN_1880,1880,ALB,0.9,Peyton L Livick,Peyton,L,Livick,NaN,...,LAVAC,L120,W,NaN,NaN,False,NaN,FC1880-253,Peyton L Livick (M / W born 1880 in Alb). Pare...,2026-06-10T18:18:22.521717+00:00
41,ALB-CN-1880-13770,ALB_CN_1880,1880,ALB,0.9,Addison Mapie,Addison,NaN,Mapie,NaN,...,MAPY,M100,B,AGRICULTURE,NaN,True,NaN,FC1880-2441,Addison Mapie (M / B born 1845 in Alb). Spouse...,2026-06-10T18:19:25.450725+00:00
42,ALB-CN-1880-13771,ALB_CN_1880,1880,ALB,0.9,Sarah J Mapie,Sarah,J,Mapie,NaN,...,MAPY,M100,B,DOMESTIC,NaN,False,NaN,FC1880-2441,Sarah J Mapie (F / B born 1851 in Alb). Spouse...,2026-06-10T18:19:25.450725+00:00
43,ALB-CN-1880-13772,ALB_CN_1880,1880,ALB,0.9,Joseph Mapie,Joseph,NaN,Mapie,NaN,...,MAPY,M100,B,AGRICULTURE,NaN,False,NaN,FC1880-2441,Joseph Mapie (M / B born 1868 in Alb). Parents...,2026-06-10T18:19:25.450725+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118031,ALB-CN-1880-28629,ALB_CN_1880,1880,ALB,0.9,George F Sheets,George,F,Sheets,NaN,...,SAT,S320,W,LABORER,NaN,False,NaN,FC1880-4968,GEORGE F SHEETS is a White male born in 1845 w...,2026-06-10T18:20:44.716359+00:00
118032,ALB-CN-1880-2863,ALB_CN_1880,1880,ALB,0.9,Samuel H More,Samuel,H,More,NaN,...,MARA,M600,B,NaN,NaN,False,NaN,FC1880-516,SAMUEL H MORE is a Black male born in 1866 wit...,2026-06-10T18:18:30.273394+00:00
118033,ALB-CN-1880-28630,ALB_CN_1880,1880,ALB,0.9,Sarah Bryan,Sarah,NaN,Bryan,NaN,...,BRYAN,B650,W,DOMESTIC,NaN,True,NaN,FC1880-4969,SARAH BRYAN is a White female born in 1820. Sh...,2026-06-10T18:20:44.716359+00:00
118034,ALB-CN-1880-28631,ALB_CN_1880,1880,ALB,0.9,Martha Bryan,Martha,NaN,Bryan,NaN,...,BRYAN,B650,W,DOMESTIC,NaN,False,NaN,FC1880-4969,MARTHA BRYAN is a White female born in 1853 wi...,2026-06-10T18:20:44.716359+00:00


In [5]:
mentions.shape

(59287, 28)

## Data Cleaning 

To use the splink dataset there are a few prerequisites that must be in place to use the splink dataset 
- Each input dataset must have a unique ID column, which is unique within the dataset. By default, Splink assumes this column will be called unique_id 

    --> this requires engineering a new data column named 'unique_id'
- Input datasets must be conformant, meaning they share the same column names and data formats 

    --> To do this going to do a Dedupe Only splink model (analyzes and looks for matching records within the same dataframe)
- Ensure data consistency by cleaning your data. This process includes standardizing date formats, matching text case, and handling invalid data 

    --> this requires standardization 
- Ensure null values (or other 'not known' indicators) are represented as true nulls, not empty strings. Splink treats null values differently from empty strings, so using true nulls guarantees proper matching across datasets 

    --> requires that i go through the data and ensure everything is Nan

Other suggestions SPLINK recommends: 
- Trim leading and trailing whitespace from string values (e.g., " john smith " becomes "john smith")
- Remove special characters from string values (e.g., "O'Hara" becomes "Ohara")
- Standardise date formats as strings in "yyyy-mm-dd" format
- Replace abbreviations with full words (e.g., standardize "St." and "Street" to "Street").

In [6]:
# first going to rename the mentions_id column to unique_id
mentions = mentions.rename(columns={"mention_id": "unique_id"})

In [7]:
id_counts = mentions['unique_id'].value_counts()
list(id_counts)

[3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,


In [8]:
id_counts

unique_id
ALB-CN-1880-17427    3
ALB-CN-1880-16388    3
ALB-CN-1880-16389    3
ALB-CN-1880-1639     3
ALB-CN-1880-16390    3
                    ..
ALB-CN-1870-4624     1
ALB-CN-1870-4625     1
ALB-CN-1870-4626     1
ALB-CN-1870-4627     1
ALB-CN-1870-4611     1
Name: count, Length: 41294, dtype: int64

In [9]:
mentions[mentions['unique_id'] == 'ALB-CN-1880-29510']

,unique_id,source,source_year,county,confidence,full_name,first_name,middle_name,last_name,maiden_name,...,nysiis_last_name,soundex_last_name,norm_race,norm_occupation,location_id,head,household_id,family_id,narrative,created
44848,ALB-CN-1880-29510,ALB_CN_1880,1880,ALB,0.9,Robert Horn,Robert,NaN,Horn,NaN,...,HARN,H650,W,AGRICULTURE,NaN,False,NaN,FC1880-5121,Robert Horn (M / W born 1863 in Alb). Occupati...,2026-06-10T18:20:49.582614+00:00
54259,ALB-CN-1880-29510,ALB_CN_1880,1880,ALB,0.9,Robert Horn,Robert,NaN,Horn,NaN,...,HARN,H650,W,AGRICULTURE,NaN,False,NaN,FC1880-5121,Robert Horn (M / W born 1863 in Alb). Occupati...,2026-06-10T18:20:49.582614+00:00
80588,ALB-CN-1880-29510,ALB_CN_1880,1880,ALB,0.9,Robert Horn,Robert,NaN,Horn,NaN,...,HARN,H650,W,AGRICULTURE,NaN,False,NaN,FC1880-5121,Robert Horn (M / W born 1863 in Alb). Occupati...,2026-06-10T18:20:49.582614+00:00


In [10]:
mentions[mentions['unique_id'] == 'ALB-CN-1880-7170']

,unique_id,source,source_year,county,confidence,full_name,first_name,middle_name,last_name,maiden_name,...,nysiis_last_name,soundex_last_name,norm_race,norm_occupation,location_id,head,household_id,family_id,narrative,created
48031,ALB-CN-1880-7170,ALB_CN_1880,1880,ALB,0.9,Rebecca Lissley,Rebecca,NaN,Lissley,NaN,...,LASLAY,L240,W,DOMESTIC,NaN,False,NaN,FC1880-1283,REBECCA LISSLEY is a White female born in 1835...,2026-06-10T18:18:51.733662+00:00
57442,ALB-CN-1880-7170,ALB_CN_1880,1880,ALB,0.9,Rebecca Lissley,Rebecca,NaN,Lissley,NaN,...,LASLAY,L240,W,DOMESTIC,NaN,False,NaN,FC1880-1283,REBECCA LISSLEY is a White female born in 1835...,2026-06-10T18:18:51.733662+00:00
83771,ALB-CN-1880-7170,ALB_CN_1880,1880,ALB,0.9,Rebecca Lissley,Rebecca,NaN,Lissley,NaN,...,LASLAY,L240,W,DOMESTIC,NaN,False,NaN,FC1880-1283,REBECCA LISSLEY is a White female born in 1835...,2026-06-10T18:18:51.733662+00:00


In [11]:
mentions[mentions['unique_id'] == 'ALB-CN-1880-7161']
    

,unique_id,source,source_year,county,confidence,full_name,first_name,middle_name,last_name,maiden_name,...,nysiis_last_name,soundex_last_name,norm_race,norm_occupation,location_id,head,household_id,family_id,narrative,created
48019,ALB-CN-1880-7161,ALB_CN_1880,1880,ALB,0.9,James Lissley,James,NaN,Lissley,NaN,...,LASLAY,L240,W,AGRICULTURE,NaN,True,NaN,FC1880-1282,JAMES LISSLEY is a White male born in 1841. He...,2026-06-10T18:18:51.733662+00:00
57430,ALB-CN-1880-7161,ALB_CN_1880,1880,ALB,0.9,James Lissley,James,NaN,Lissley,NaN,...,LASLAY,L240,W,AGRICULTURE,NaN,True,NaN,FC1880-1282,JAMES LISSLEY is a White male born in 1841. He...,2026-06-10T18:18:51.733662+00:00
83759,ALB-CN-1880-7161,ALB_CN_1880,1880,ALB,0.9,James Lissley,James,NaN,Lissley,NaN,...,LASLAY,L240,W,AGRICULTURE,NaN,True,NaN,FC1880-1282,JAMES LISSLEY is a White male born in 1841. He...,2026-06-10T18:18:51.733662+00:00


### Duplicate People in the Same Census
Something that I realized was there were duplicate people in the same census. I am going to remove these duplicates. 

In [12]:
mentions.shape
prev_rows = mentions.shape[0]

In [13]:
# removing duplicates in the census 
mentions = mentions.drop_duplicates(subset=['unique_id'])

In [14]:
mentions.shape
new_rows = mentions.shape[0]
dropped_rows = prev_rows - new_rows

In [15]:
# going to calculate the number of dropped duplicates 
print(f"The number of dropped duplicates is {dropped_rows}")

The number of dropped duplicates is 17993


Going to drop the following columns:
    - mention_id

    - source

    - confidence 

    - race 

    - occupation
    
    - created

    - full_name 

    - first_name

    - last_name

Need to change birth years to int

Going to normalize middle_name and maiden_name

In [16]:
# going to examine all the columns in the dataset 
for col in mentions.columns:
    print(col)

unique_id
source
source_year
county
confidence
full_name
first_name
middle_name
last_name
maiden_name
birth_year
death_year
race
gender
occupation
legal_status
is_enslaver
norm_first_name
nysiis_last_name
soundex_last_name
norm_race
norm_occupation
location_id
head
household_id
family_id
narrative
created


In [17]:
mentions['confidence'].value_counts(dropna=False)

confidence
0.9    41294
Name: count, dtype: int64

We see that there is no variance in confidence for every entry present in the dataset. This column isn't informative and will be dropped as well.

In [18]:
mentions = mentions.drop(columns=['source', 'confidence', 'race', 'occupation', 'created','full_name','first_name','last_name'])
mentions.head()

,unique_id,source_year,county,middle_name,maiden_name,birth_year,death_year,gender,legal_status,is_enslaver,norm_first_name,nysiis_last_name,soundex_last_name,norm_race,norm_occupation,location_id,head,household_id,family_id,narrative
39,ALB-CN-1880-27906,1880,ALB,M,NaN,1876.0,NaN,F,F,False,ANN,STACLAY,S324,W,DOMESTIC,NaN,False,NaN,FC1880-4835,Anna M Stickley (F / W born 1876 in Alb). Pare...
40,ALB-CN-1880-1377,1880,ALB,L,NaN,1880.0,NaN,M,F,False,PEYTON,LAVAC,L120,W,NaN,NaN,False,NaN,FC1880-253,Peyton L Livick (M / W born 1880 in Alb). Pare...
41,ALB-CN-1880-13770,1880,ALB,NaN,NaN,1845.0,NaN,M,F,False,ADDISON,MAPY,M100,B,AGRICULTURE,NaN,True,NaN,FC1880-2441,Addison Mapie (M / B born 1845 in Alb). Spouse...
42,ALB-CN-1880-13771,1880,ALB,J,NaN,1851.0,NaN,F,F,False,SARAH,MAPY,M100,B,DOMESTIC,NaN,False,NaN,FC1880-2441,Sarah J Mapie (F / B born 1851 in Alb). Spouse...
43,ALB-CN-1880-13772,1880,ALB,NaN,NaN,1868.0,NaN,M,F,False,JOSEPH,MAPY,M100,B,AGRICULTURE,NaN,False,NaN,FC1880-2441,Joseph Mapie (M / B born 1868 in Alb). Parents...


In [19]:
mentions['maiden_name'].value_counts(dropna=False)

maiden_name
NaN    41294
Name: count, dtype: int64

In [20]:
mentions['middle_name'].value_counts(dropna=False)

middle_name
NaN        26622
E           1487
A           1486
W           1029
M            993
           ...  
Gooloe         1
Watkins        1
Handy          1
Howe           1
Trivy          1
Name: count, Length: 460, dtype: int64

In [21]:
mentions['middle_name'].describe()

count     14672
unique      459
top           E
freq       1487
Name: middle_name, dtype: object

In [22]:
# going to normalize the middle_name column by converting to all capital letters
mentions['middle_name'] = mentions['middle_name'].str.upper()

In [23]:
mentions['middle_name'].value_counts()

middle_name
E           1487
A           1486
W           1029
M            993
J            988
            ... 
SINK           1
SHERIDAN       1
GURTRUDE       1
MM             1
MH             1
Name: count, Length: 452, dtype: int64

In [24]:
mentions

,unique_id,source_year,county,middle_name,maiden_name,birth_year,death_year,gender,legal_status,is_enslaver,norm_first_name,nysiis_last_name,soundex_last_name,norm_race,norm_occupation,location_id,head,household_id,family_id,narrative
39,ALB-CN-1880-27906,1880,ALB,M,NaN,1876.0,NaN,F,F,False,ANN,STACLAY,S324,W,DOMESTIC,NaN,False,NaN,FC1880-4835,Anna M Stickley (F / W born 1876 in Alb). Pare...
40,ALB-CN-1880-1377,1880,ALB,L,NaN,1880.0,NaN,M,F,False,PEYTON,LAVAC,L120,W,NaN,NaN,False,NaN,FC1880-253,Peyton L Livick (M / W born 1880 in Alb). Pare...
41,ALB-CN-1880-13770,1880,ALB,NaN,NaN,1845.0,NaN,M,F,False,ADDISON,MAPY,M100,B,AGRICULTURE,NaN,True,NaN,FC1880-2441,Addison Mapie (M / B born 1845 in Alb). Spouse...
42,ALB-CN-1880-13771,1880,ALB,J,NaN,1851.0,NaN,F,F,False,SARAH,MAPY,M100,B,DOMESTIC,NaN,False,NaN,FC1880-2441,Sarah J Mapie (F / B born 1851 in Alb). Spouse...
43,ALB-CN-1880-13772,1880,ALB,NaN,NaN,1868.0,NaN,M,F,False,JOSEPH,MAPY,M100,B,AGRICULTURE,NaN,False,NaN,FC1880-2441,Joseph Mapie (M / B born 1868 in Alb). Parents...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108224,ALB-CN-1880-16337,1880,ALB,NaN,NaN,1860.0,NaN,M,F,False,LEWIS,NALSAN,N425,B,LABORER,NaN,True,NaN,FC1880-2891,LEWIS NELSON is a Black male born in 1860. He ...
108225,ALB-CN-1880-16338,1880,ALB,NaN,NaN,1820.0,NaN,M,F,False,GEORGE,KANADY,K530,W,AGRICULTURE,NaN,True,NaN,FC1880-2891,GEORGE KENNEDY is a White male born in 1820 Hi...
108227,ALB-CN-1880-16339,1880,ALB,NaN,NaN,1824.0,NaN,F,F,False,ELIZABETH,KANADY,K530,W,DOMESTIC,NaN,False,NaN,FC1880-2891,ELIZA KENNEDY is a White female born in 1824. ...
108228,ALB-CN-1880-1634,1880,ALB,W,NaN,1856.0,NaN,F,F,False,MARTHA,GADNAN,G355,W,DOMESTIC,NaN,False,NaN,FC1880-304,MATTIE W GOODMAN is a White female born in 185...


In [25]:
# looking at what datatype birth_year is 
print(mentions['birth_year'].dtype)
print(mentions['death_year'].dtype)

float64
float64


In [26]:
# looking at value_counts
mentions['birth_year'].value_counts(dropna=False)

birth_year
1869.0    1175
1868.0    1156
1879.0    1084
1867.0    1067
1862.0    1028
          ... 
1781.0       1
1775.0       1
1772.0       1
1773.0       1
1768.0       1
Name: count, Length: 113, dtype: int64

In [27]:
mentions['birth_year'].describe()

count    41278.000000
mean      1852.774165
std         19.314895
min       1765.000000
25%       1841.000000
50%       1857.000000
75%       1868.000000
max       1880.000000
Name: birth_year, dtype: float64

In [28]:
for col in mentions.columns:
    print("------------------------")
    print(mentions[col].value_counts(dropna=False))

------------------------
unique_id
ALB-CN-1880-16340    1
ALB-CN-1880-27906    1
ALB-CN-1880-1377     1
ALB-CN-1880-13770    1
ALB-CN-1880-13771    1
                    ..
ALB-CN-1870-19785    1
ALB-CN-1870-19786    1
ALB-CN-1870-19787    1
ALB-CN-1870-19788    1
ALB-CN-1870-19789    1
Name: count, Length: 41294, dtype: int64
------------------------
source_year
1880    27271
1870    14023
Name: count, dtype: int64
------------------------
county
ALB    41294
Name: count, dtype: int64
------------------------
middle_name
NaN         26622
E            1487
A            1486
W            1029
M             993
            ...  
SINK            1
SHERIDAN        1
GURTRUDE        1
MM              1
MH              1
Name: count, Length: 453, dtype: int64
------------------------
maiden_name
NaN    41294
Name: count, dtype: int64
------------------------
birth_year
1869.0    1175
1868.0    1156
1879.0    1084
1867.0    1067
1862.0    1028
          ... 
1781.0       1
1775.0       1
177

Upon further examination there are some other columns that are completely null. Those columns are maiden_name, is_enslaver, enslaver_id, location_id, legal_status, and death_year. It is not surprising that this information is missing due to both of these censuses occuring after the abolition of slavery. These columns will be very useful once we move to matching records before and after slavery abolition. For now I will also drop these columns for the splink record matching. 

In [29]:
# dropping additional columns
mentions = mentions.drop(columns=['maiden_name','death_year','legal_status','is_enslaver','location_id'])

In [30]:
# before we can begin any EDA we must split the dataset into two - 1870 and 1880
mentions_1870 = mentions[mentions['source_year'] == 1870]
mentions_1880 = mentions[mentions['source_year'] == 1880]

In [31]:
print(mentions_1870.shape)
print(mentions_1880.shape)

(14023, 15)
(27271, 15)


### Cleaning Summary:
- Renaming the mention_id column to unique_id
- Removing the following columns 
    - Source: The name of the source in which the row data originated from
        - Why: Not informative for matching records
    - Original_data: The file's entire row as a JSON object
        - Why: Not informative for matching records
    - Confidence: The confidence of the transcription of the data from the primary document ?...
        - Why: Everything is 0.9; Not informative when trying to match records 
    - Race: The race of the person. B, W, M, I, C or Y.
        - Why: There is a normalized race column that is cleaner and more informative
    - Occupation: The work role of the person
        - Why: There is a normalized column of occupation that is more informative
    - Created: The timestamp and IP location for when the assertion(the entire row) was created 
        - Why: Not informative for matching records
    - full_name: The combination of the first-name, the middle_name, and the last_name separated by spaces
        - Why: there is normalized data that is more informative 
    - first_name: The given name 
        - Why: There is a normalized version
    - last_name:
        -Why: There is a normalized version
- Normalized the middle_name column 
    - Simply made everything uppercase 
- Removed variables that are completely blank now due to when these censuses occured (but should keep once we start matching records pre and post civil war)
    - legal_status - 100% full but everything is 'F' so not the most informative
    - is_enslaver - 100% full but everything is False
    - location_id - 100% full but everything is NaN
- Removed the following columns as well 
    - maiden_name 
    - death_year 
- Split the mentions data into 1870 and 1880
    - dont want to compare within a census so its important that they are split into multiples
- Removed duplicate_ids

## Exploratory Analysis with Splink
### Goals: 

    1. Analyse missingness  

    2. Analyse the distribution of values in your data

In [32]:
from splink.exploratory import completeness_chart
from splink import DuckDBAPI

In [33]:
mentions_1880.columns

Index(['unique_id', 'source_year', 'county', 'middle_name', 'birth_year',
       'gender', 'norm_first_name', 'nysiis_last_name', 'soundex_last_name',
       'norm_race', 'norm_occupation', 'head', 'household_id', 'family_id',
       'narrative'],
      dtype='object')

In [34]:
completeness_chart(
    [mentions_1870, mentions_1880],
    cols=['unique_id', 'source_year', 'county', 'middle_name', 'birth_year',
       'gender', 'norm_first_name', 'nysiis_last_name', 'soundex_last_name',
       'norm_race', 'norm_occupation', 'head', 'household_id', 'family_id',
       'narrative'],
    db_api=DuckDBAPI(),
    table_names_for_chart=["1870 Ablemarle County Census", "1880 Ablemarle County Census"],
)

alt.LayerChart(...)

### Analyzing Figure 1 
The previous figure is an image that depicts the percentage completeness of each column in the 1870 and 1880 separated census data.

### 1870 Interpretation: 

- There are 12 columns in which there are virtually no data missing. Those columns are: 'unique_id', 'source_year', 'county', 'norm_race', 'head', 'family_id', 'gender', 'soundex_last_name', 'nysiis_last_name', 'birth_year', 'norm_first_name', 'narrative', and 'household_id'. 

- Norm_occupation is missing 35% of its data

- Middle_name is missing 73% of its data

### 1880 Interpretation:

- There are 11 columns in which there are virtually no data missing. Those columns are: 'unique_id', 'source_year', 'county', 'norm_race', 'head', 'family_id', 'gender', 'soundex_last_name', 'nysiis_last_name', 'birth_year', 'norm_first_name', and 'narrative'. 

- Norm_occupation is missing 23% of its data

- Middle_name is missing 60% of its data

- Household_id is completely empty

### Interpreting the Visualizations Together:

- There is an increase in column completeness from 1870 to 1880 for norm_occupation (7% more respondents recorded their occupation)
- The household_id is completely empty for the 1880 census '
- There is a increase in colun completeness from 1870 to 1880 for middle_name (13%)


In [35]:
from splink.exploratory import profile_columns

In [36]:
profile_columns(mentions_1870, db_api=DuckDBAPI(), top_n=10, bottom_n=5)

alt.VConcatChart(...)

### Analyzing the Columns in the 1870 Census 
- The most common entries for middle name in 1870 are initials
- There are slightly more females recorded in the census; Females make up 51.14% and Males make up 48.86%
- There are slightly more recorded Black individuals than white; 17.24% difference

In [37]:
profile_columns(mentions_1880, db_api=DuckDBAPI(), top_n=10, bottom_n=5)

alt.VConcatChart(...)

### Analyzing the Columns in the 1880 Census
- It is still common to just report the middle initial 
- There are still slightly more Females; Females make up 52.16% and Males make up 47.84% of the data
- There are slightly more recorded Black individuals than white

Overall: there are no significant shifts in data distributions from the 1870 to the 1880 census

## Blocking

 We rely on blocking rules, which specify which pairwise comparisons to generate because it is not advantageous if we compare every single row to every other row. This is difficult because as your data increases the comparisons will also increase quadratically. 

    1. Use blocking rules to generate candidate pairwise record comparisons
    2. Use a probabilistic linkage model to score these candidate pairs, to determine which ones should be linked

Blocking rules are the most important determinant of the performance of your linkage job. Its important that the blocking rules are not too lenient (will make incorrect matches) or too harsh (can miss valid matches).








In [38]:
from splink.blocking_analysis import count_comparisons_from_blocking_rule

In [39]:
mentions.columns

Index(['unique_id', 'source_year', 'county', 'middle_name', 'birth_year',
       'gender', 'norm_first_name', 'nysiis_last_name', 'soundex_last_name',
       'norm_race', 'norm_occupation', 'head', 'household_id', 'family_id',
       'narrative'],
      dtype='object')

In [40]:
# lets calculate the total number of comparisons that must occur before any blocking can occur
total_comparisons = len(mentions_1870) * len(mentions_1880)
print(len(mentions_1870))
print(len(mentions_1880))
print(f"The total number of comparisons that must occur before any blocking occurs is {total_comparisons}")


14023
27271
The total number of comparisons that must occur before any blocking occurs is 382421233


In [41]:
# going to analyze some of the generated counts from different bloocking rules 
from splink import block_on

br = [block_on("norm_first_name", "nysiis_last_name"),block_on("gender", "nysiis_last_name"),block_on("norm_race", "nysiis_last_name", "gender"),block_on("norm_race", "norm_first_name", "gender"),block_on("gender","norm_race")]

for rule in br:
    counts = count_comparisons_from_blocking_rule(
    table_or_tables=[mentions_1870,mentions_1880], # what data to perform the record linkage on
    blocking_rule=rule, # what rules must be followed to generate candidate pairwise connections
    link_type="link_only", # doing dedupe_only because I am providing a single input table; looking for duplicates in the table essentially
    db_api=DuckDBAPI() # api access
)
    print(counts)
    print('----------')

{'number_of_comparisons_generated_pre_filter_conditions': 13167, 'number_of_comparisons_to_be_scored_post_filter_conditions': 13167, 'filter_conditions_identified': '', 'equi_join_conditions_identified': 'l."norm_first_name" = r."norm_first_name" AND l."nysiis_last_name" = r."nysiis_last_name"', 'link_type_join_condition': 'where l."source_dataset" || \'-__-\' || l."unique_id" < r."source_dataset" || \'-__-\' || r."unique_id" and l."source_dataset" != r."source_dataset"'}
----------
{'number_of_comparisons_generated_pre_filter_conditions': 475786, 'number_of_comparisons_to_be_scored_post_filter_conditions': 475786, 'filter_conditions_identified': '', 'equi_join_conditions_identified': 'l."gender" = r."gender" AND l."nysiis_last_name" = r."nysiis_last_name"', 'link_type_join_condition': 'where l."source_dataset" || \'-__-\' || l."unique_id" < r."source_dataset" || \'-__-\' || r."unique_id" and l."source_dataset" != r."source_dataset"'}
----------
{'number_of_comparisons_generated_pre_fi

### Interpreting the Previous Output

What does each variable represent? 

- number_of_comparisons_generated_pre_filter_conditions: how many rows meet the blocking rule conditions

- number_of_comparisons_to_be_scored_post_filter_conditions: how many actual comparisons must occur (removes self-comparions and duplicate symmetric pairs)

- both of these numbers will be the same because duplicate symmetric comparisons do not occur with link_only record linkage

### Interpreting each output:

Before any Blocking: 

- Total number of comparisons that must occur is 382,421,233 

    - How do you get this number? You simply multiply all the records in 1 census by all the records in the second census, thus you multiply 14,023 by 27,271.

When Blocking on First and Last Name:

- There are 13,167 comparisons that must occur. 

- There is a 99.99% reduction in the number of comparisons that must occur (to no blocking rules). 

When Blocking on Gender and Last Name: 

- There are 475,786 comparisons that must occur.

- There is a 99.86% reduction in the number of comparisons that must occur (to no blocking rules). 

When Blocking on Race, Last Name, and Gender:

- There are 262,875 comparisons that must occur.

- There is a 99.93% reduction in the number of comparisons that must occur (to no blocking rules). 

When Blocking on Gender and Race:

- There are 87,231,498 comparisons that must occur.

- There is a 77.19% reduction in the number of comparisons that must occur (to no blocking rules).

Reduction Percentage Formula: 1 - ( Number of comparisons generated by blocking / Total Number of Comparisons )


### Key Takeaways:

- The most restrictive blocking rule is blocking on an exact first and last name. On the other hand, the least restrictive is blocking only on gender and last name. The blocking rule that is the middle ground is last name, gender, and race. The middle ground still requires 1.8 million comparisons. 
- However, blocking on last name can be a difficult path to choose. Why? Because African Americans didn't have a last name prior to the civil war and many women age out of their last name. Going to focus on blocking on gender and race 

In [42]:
# lets look into some of these numbers and get some confirmation 

name_keys = ["norm_race", "gender"]

g1870 = (
    mentions_1870 # looks at the 1870 data
    .groupby(name_keys) # groups the data by the first and last name
    .size() # gets the count for each grouping
    .reset_index(name="n_1870") # sets the index name to n_1870
)

g1880 = (
    mentions_1880 # looks at the 1880 data
    .groupby(name_keys) # groups the data by the first and last name
    .size() # gets the count for each group
    .reset_index(name="n_1880") # sets the index name to n_1880
)


merged = g1870.merge(g1880, on=name_keys, how="inner") # merging the data based on their name keys; inner join only returns data that is matching in both 1870 and 1880

# calculate the comparisons
merged["comparisons"] = merged["n_1870"] * merged["n_1880"]

print("Total comparisons:", merged["comparisons"].sum())

Total comparisons: 87231498


In [43]:
merged

,norm_race,gender,n_1870,n_1880,comparisons
0,B,F,4285,3504,15014640
1,B,M,3934,3167,12458978
2,W,F,2886,10720,30937920
3,W,M,2917,9880,28819960


### Visualization Interpretation 
When blocking on race and gender, the comparisons can be broken down into four primary groups. Those four groups are black males, white males, black females, and white females. 

### Finding 'worst offending' values for the blocking rule
This investigates what values under certain blocking rules will generate the most comparisons

In [44]:
from splink.blocking_analysis import n_largest_blocks

result = n_largest_blocks(    
    table_or_tables=[mentions_1870,mentions_1880],
    blocking_rule= block_on("norm_race", "gender"),
    link_type="link_only",
    db_api=DuckDBAPI(),
    n_largest=10
    )

result.as_pandas_dataframe()

,key_0,key_1,count_l,count_r,block_count
0,W,F,2886,10720,30937920
1,W,M,2917,9880,28819960
2,B,F,4285,3504,15014640
3,B,M,3934,3167,12458978


In [45]:
from splink.blocking_analysis import n_largest_blocks

result = n_largest_blocks(    
    table_or_tables=[mentions_1870,mentions_1880],
    blocking_rule= block_on("norm_race", "gender", "norm_first_name"),
    link_type="link_only",
    db_api=DuckDBAPI(),
    n_largest=10
    )

result.as_pandas_dataframe()

,key_0,key_1,key_2,count_l,count_r,block_count
0,W,F,MARY,350,1414,494900
1,W,M,JOHN,267,1106,295302
2,W,M,WILLIAM,285,1035,294975
3,B,F,MARY,346,378,130788
4,W,M,JAMES,181,699,126519
5,W,F,ELIZABETH,168,651,109368
6,B,M,JOHN,325,279,90675
7,B,M,WILLIAM,279,298,83142
8,W,F,SARAH,148,547,80956
9,W,M,CHARLES,125,477,59625


### Interpreting the Previous Output

The previous code is meant to visualize how certain blocking rules do not have the same distribution of comparisons across groupings. This visualizations highlight what certain groupings under a set of blocking rules. The dataframe depicts the largest block comparisons that must occur under the blocking rule that the gender and race are the exact same. What you see is that white females generate the largest proportion of the comparisons. The second group that requires the most comparisons are white males. 



The second dataframe does the same thing but under the blocking rule that the race, gender and first name are the exact same. The grouping that requires the most comparisons in decending order are white females named Mary, white males named John, and then white males named William.


### Counting the number of comparisons by blocking rule

In [46]:
from splink.blocking_analysis import (
    cumulative_comparisons_to_be_scored_from_blocking_rules_chart,
)


blocking_rules_for_analysis = [
    block_on("norm_race", "gender"),
    block_on("norm_race", "gender", "norm_first_name")
            
]


cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables=[mentions_1870,mentions_1880],
    blocking_rules=blocking_rules_for_analysis,
    db_api=DuckDBAPI(),
    link_type="link_only",
)

alt.Chart(...)

### Interpreting the Previous Visualization

- When blocking on race, gender and exact first name you generate 2.4 million comparisons 
- When blocking on race and gender you generate 87 million comparisons

### Key Takeaways 
The previous figure is another way to interactively visualizae the number of comparions that will occur using different blocking rules.


When Blocking on First and Last Name: 13.2K Comparisons

When Blocking on Gender and Last Name: 475K Comparisons

When Blocking on Race, Last Name, and Gender: 262.9K Comparisons

When Blocking on Gender and Race: 87.2M Comparisons





### Overall Ranking Based on the Restrictiveness of the Blocking Rules (Ascending Order):

- First and Last Name: 13.2K Comparisons

- Race, Last Name, and Gender: 262.9K Comparisons

- Gender and Last Name: 475K Comparisons

- No Blocking Rules: 382.4M Comparisons


### Approach to the Splink Model 

- Will decrease the number of overall comparisons that must occur by first blocking on exact race and gender and then creating levels based on the first name and birth year



# Estimating Model Parameters 

The purpose of the next section is to estimate a probabilistic linkage model to score each of the comparisons. The result of this model is a match score on if the two records are the same person. The match weights can determine how much of an influence certain information is in determining if two records are a match. The match weights are are derived from the m and u parameters of the underlying Fellegi Sunter model.

In [47]:
mentions_1870.columns

Index(['unique_id', 'source_year', 'county', 'middle_name', 'birth_year',
       'gender', 'norm_first_name', 'nysiis_last_name', 'soundex_last_name',
       'norm_race', 'norm_occupation', 'head', 'household_id', 'family_id',
       'narrative'],
      dtype='object')

### Setting up the Linkage Model

### Creating custom levels for birth year and first name

This has the purpose of allowing splink to break a data field's similarity into discrete categries. Doing custom to better account for the messiness of the census data

In [48]:
import splink.comparison_library as cl
from splink import Linker, SettingsCreator, block_on, DuckDBAPI
from splink import comparison_library as cl 
from splink import comparison_level_library as cll

#creating custom levels for birth_year
birth_year_comparison = cl.CustomComparison(
    output_column_name = "birth_year",
    comparison_levels = [
        cll.NullLevel("birth_year"),
        cll.CustomLevel(
            "abs(birth_year_l - birth_year_r) = 0",
            label_for_charts = "Exact Birth Year"),
        cll.CustomLevel(
            "abs(birth_year_l - birth_year_r) <= 1",
            label_for_charts = "Within 1 Year"),
        cll.CustomLevel(
            "abs(birth_year_l - birth_year_r) <= 2",
            label_for_charts = "Within 2 Years"),
        cll.CustomLevel(
            "abs(birth_year_l - birth_year_r) <= 5",
            label_for_charts = "Within 5 Years"),
        cll.ElseLevel()
          
    ]
)

# the ordering of the following matters so john and joan would fall into "within 1 letter" rather than "same inital letter"
first_name_comparison = cl.CustomComparison(
    output_column_name = "norm_first_name",
    comparison_levels = [
        cll.NullLevel("norm_first_name"),
        cll.CustomLevel(
            "levenshtein(norm_first_name_l, norm_first_name_r) = 0",
            label_for_charts = "Exact First Name"),
        cll.CustomLevel(
            "levenshtein(norm_first_name_l, norm_first_name_r) = 1",
            label_for_charts = "Within 1 Letter"),
        cll.CustomLevel(
            "levenshtein(norm_first_name_l, norm_first_name_r) = 2",
            label_for_charts = "Within 2 Letters"),
        cll.CustomLevel(
            "substr(norm_first_name_l, 1, 1) = substr(norm_first_name_r, 1, 1)",
            label_for_charts = "Same First Initial"),
        cll.ElseLevel()
          
    ]
)




In [49]:
# the following code simply specifies the linkage model
# settings = SettingsCreator(
#     link_type="link_only",
#     comparisons=[                               # everything in the comparisons is what is used to determine the match score
#         cl.ExactMatch("gender"),   
#         cl.ExactMatch("norm_race"),
#         cl.NameComparison("nysiis_last_name"), # there are 5 default comparison levels : exact match, null match, and 3 different levels based on the jaro-winkler similarity
#         birth_year_comparison,            # 6 custom comparison levels
#         first_name_comparison             # 5 custom comparison level
#     ],
#     blocking_rules_to_generate_predictions=[ #  will only look at records in which the race gender are the same 
#         block_on("gender","norm_race") # potential to insert multi-pass blocking HERE; running multiple iterations; also block on absolute value of birth year 
#     ],
#     retain_intermediate_calculation_columns=True,   # the runtime for the computations will be longer but will output more information so that i can understand the calculations
# )



settings = {
    "link_type": "link_only",

    "blocking_rules_to_generate_predictions": [
        "l.norm_first_name = r.norm_first_name and l.nysiis_last_name = r.nysiis_last_name",
        "l.norm_race = r.norm_race and l.gender = r.gender and abs(l.birth_year - r.birth_year) <= 5",
    ],

    "comparisons": [
    birth_year_comparison,
    cl.ExactMatch("norm_race"),
    cl.ExactMatch("gender"),
    first_name_comparison,
]
}


linker = Linker([mentions_1870, mentions_1880], settings, db_api=DuckDBAPI())

# ADD INTERMEDIATE STEP TO VISUALIZE THE DATA AFTER THE MULTI-PASS BLOCKING

### Estimating the probability_two_random_records_match

### Key Information

probability_two_random_records_match parameter is the probability that two records taken at random from your input data represent a match (typically a very small number)
The u values are the proportion of records falling into each ComparisonLevel amongst truly non-matching records.
The m values are the proportion of records falling into each ComparisonLevel amongst truly matching records

In [50]:
mentions_1870.columns

Index(['unique_id', 'source_year', 'county', 'middle_name', 'birth_year',
       'gender', 'norm_first_name', 'nysiis_last_name', 'soundex_last_name',
       'norm_race', 'norm_occupation', 'head', 'household_id', 'family_id',
       'narrative'],
      dtype='object')

In [51]:
deterministic_rules = [
   block_on("gender","norm_race", "norm_first_name")
]

linker.training.estimate_probability_two_random_records_match(deterministic_rules, recall=0.90)
# room for ground truth; look at metrics for the gorund truth data and see how can use this for the recall metric 

Probability two random records match is estimated to be  0.00749.
This means that amongst all possible pairwise record comparisons, one in 133.54 are expected to match.  With 382,421,233 total possible comparisons, we expect a total of around 2,863,731.11 matching pairs


You want your deterministic rules to be more restricive than your blocking rules so that you are able to get a more representative estimate of the probability. I chose to require that gender, race, and first name were the exact same. Lastly, I estimated that this rules would cover about 90% of true record matches

Match weights depend on the U and M values 

- U represents the probability that a specific field agrees given that the records are not a match. For example, what is the probability that two individuals, one black and one white, share the same birth year? What about sharing the same name with a levenshtein difference of 2 (robin vs rob)?; think of it as the probability of coincidences; comparison levels are very important for this 

- M represents the probability that a specific field agrees given that the records are a match

### Estimating U probabilities 

The U probability represents the proportion of records falling into each Comparison Level amongst truly non-matching records. This is estimated by sampling random pairs of records and computing the distribution for each comparison level variable. The intuition is that most of the randomly sampled records will not be a match, thus you can get a better feel for the distirbution of the comparison levels

In [52]:
linker.training.estimate_u_using_random_sampling(max_pairs=1e6)

You are using the default value for `max_pairs`, which may be too small and thus lead to inaccurate estimates for your model's u-parameters. Consider increasing to 1e8 or 1e9, which will result in more accurate estimates, but with a longer run time.
----- Estimating u probabilities using random sampling -----

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - birth_year (no m values are trained).
    - norm_race (no m values are trained).
    - gender (no m values are trained).
    - norm_first_name (no m values are trained).


### Estimating M Probabilities 
Must be estimated via Expectation Maximisation

- Generates pairwise record comparisons and uses them to maximise a likelihood function

- Each estimation pass requires the user to configure an estimation blocking rule to reduce the number of record comparisons generated to a manageable level


In [53]:
mentions_1870.columns

Index(['unique_id', 'source_year', 'county', 'middle_name', 'birth_year',
       'gender', 'norm_first_name', 'nysiis_last_name', 'soundex_last_name',
       'norm_race', 'norm_occupation', 'head', 'household_id', 'family_id',
       'narrative'],
      dtype='object')

In [54]:
training_blocking_rule = block_on("gender")
training_session_gender_race = (
    linker.training.estimate_parameters_using_expectation_maximisation(training_blocking_rule)
)


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l."gender" = r."gender"

Parameter estimates will be made for the following comparison(s):
    - birth_year
    - norm_race
    - norm_first_name

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - gender

Iteration 1: Largest change in params was -0.502 in the m_probability of birth_year, level `Exact Birth Year`
Iteration 2: Largest change in params was -0.214 in the m_probability of birth_year, level `Exact Birth Year`
Iteration 3: Largest change in params was -0.0983 in the m_probability of birth_year, level `Exact Birth Year`
Iteration 4: Largest change in params was 0.0935 in the m_probability of birth_year, level `All other comparisons`
Iteration 5: Largest change in params was 0.0918 in the m_probability of birth_year, level `All other comparisons`
Iteration 6: Largest change in params was 0.0805 in the m_probab

In [55]:
training_blocking_rule = block_on("norm_race", "norm_first_name")
training_session_race = linker.training.estimate_parameters_using_expectation_maximisation(
    training_blocking_rule
)


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l."norm_race" = r."norm_race") AND (l."norm_first_name" = r."norm_first_name")

Parameter estimates will be made for the following comparison(s):
    - birth_year
    - gender

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - norm_race
    - norm_first_name

Iteration 1: Largest change in params was -0.048 in the m_probability of gender, level `All other comparisons`
Iteration 2: Largest change in params was 0.0143 in probability_two_random_records_match
Iteration 3: Largest change in params was 0.0263 in probability_two_random_records_match
Iteration 4: Largest change in params was 0.0465 in probability_two_random_records_match
Iteration 5: Largest change in params was 0.0766 in probability_two_random_records_match
Iteration 6: Largest change in params was 0.113 in probability_two_random_records_match
Iteration 7: L

### Visualizing Model Parameters


In [56]:
linker.visualisations.match_weights_chart()

/home/ybf3jw/.conda/envs/capstone/lib/python3.10/site-packages/altair/vegalite/v6/api.py:4138: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  return _tp.from_dict(dct, validate=validate)


alt.VConcatChart(...)

### Interpreting the Above Figure

The above figure aims to depict match weight for each comparison level.

In [57]:
linker.visualisations.m_u_parameters_chart()

alt.HConcatChart(...)

In [58]:
linker.visualisations.parameter_estimate_comparisons_chart()

alt.Chart(...)

In [59]:
# settings = linker.misc.save_model_to_json(
#     '/Users/aniyahmcwilliams/Summer 2026/Capstone/linker_race_gender_birth_name.json', overwrite=True
# )

In [60]:
linker.evaluation.unlinkables_chart()

alt.LayerChart(...)

### Attempting to actually predict with the model

In [61]:
df_predictions = linker.inference.predict(threshold_match_probability=0.2)
df_predictions.as_pandas_dataframe(limit=5)

Blocking time: 3.14 seconds
Predict time: 8.63 seconds


,match_weight,match_probability,source_dataset_l,source_dataset_r,unique_id_l,unique_id_r,birth_year_l,birth_year_r,gamma_birth_year,norm_race_l,...,gamma_norm_race,gender_l,gender_r,gamma_gender,norm_first_name_l,norm_first_name_r,gamma_norm_first_name,nysiis_last_name_l,nysiis_last_name_r,match_key
0,-0.970716,0.337859,__splink__input_table_0,__splink__input_table_1,ALB-CN-1870-11817,ALB-CN-1880-31078,1851.0,1858.0,0,W,...,0,M,M,1,JOHN,JOHN,4,PANDAXTAR,PANDAXTAR,0
1,-0.970716,0.337859,__splink__input_table_0,__splink__input_table_1,ALB-CN-1870-11817,ALB-CN-1880-31081,1851.0,1879.0,0,W,...,0,M,M,1,JOHN,JOHN,4,PANDAXTAR,PANDAXTAR,0
2,-0.478523,0.417830,__splink__input_table_0,__splink__input_table_1,ALB-CN-1870-21918,ALB-CN-1880-13620,1849.0,1879.0,0,B,...,1,M,M,1,JAMES,JAMES,4,BRAC,BRAC,0
3,-0.478523,0.417830,__splink__input_table_0,__splink__input_table_1,ALB-CN-1870-14528,ALB-CN-1880-13621,1867.0,1878.0,0,B,...,1,M,M,1,ALFRED,ALFRED,4,BRAC,BRAC,0
4,-0.478523,0.417830,__splink__input_table_0,__splink__input_table_1,ALB-CN-1870-14527,ALB-CN-1880-13622,1864.0,1879.0,0,B,...,1,F,F,1,CATHARINE,CATHARINE,4,BRAC,BRAC,0


In [62]:
pd.set_option('display.max_columns',None)
df_predictions.as_pandas_dataframe(limit=5)

,match_weight,match_probability,source_dataset_l,source_dataset_r,unique_id_l,unique_id_r,birth_year_l,birth_year_r,gamma_birth_year,norm_race_l,norm_race_r,gamma_norm_race,gender_l,gender_r,gamma_gender,norm_first_name_l,norm_first_name_r,gamma_norm_first_name,nysiis_last_name_l,nysiis_last_name_r,match_key
0,-0.970716,0.337859,__splink__input_table_0,__splink__input_table_1,ALB-CN-1870-11817,ALB-CN-1880-31078,1851.0,1858.0,0,W,B,0,M,M,1,JOHN,JOHN,4,PANDAXTAR,PANDAXTAR,0
1,-0.970716,0.337859,__splink__input_table_0,__splink__input_table_1,ALB-CN-1870-11817,ALB-CN-1880-31081,1851.0,1879.0,0,W,B,0,M,M,1,JOHN,JOHN,4,PANDAXTAR,PANDAXTAR,0
2,-0.478523,0.417830,__splink__input_table_0,__splink__input_table_1,ALB-CN-1870-21918,ALB-CN-1880-13620,1849.0,1879.0,0,B,B,1,M,M,1,JAMES,JAMES,4,BRAC,BRAC,0
3,-0.478523,0.417830,__splink__input_table_0,__splink__input_table_1,ALB-CN-1870-14528,ALB-CN-1880-13621,1867.0,1878.0,0,B,B,1,M,M,1,ALFRED,ALFRED,4,BRAC,BRAC,0
4,-0.478523,0.417830,__splink__input_table_0,__splink__input_table_1,ALB-CN-1870-14527,ALB-CN-1880-13622,1864.0,1879.0,0,B,B,1,F,F,1,CATHARINE,CATHARINE,4,BRAC,BRAC,0


In [63]:
df_predictions.as_pandas_dataframe().sort_values('match_probability',ascending=False)

,match_weight,match_probability,source_dataset_l,source_dataset_r,unique_id_l,unique_id_r,birth_year_l,birth_year_r,gamma_birth_year,norm_race_l,norm_race_r,gamma_norm_race,gender_l,gender_r,gamma_gender,norm_first_name_l,norm_first_name_r,gamma_norm_first_name,nysiis_last_name_l,nysiis_last_name_r,match_key
8,-0.316599,0.445357,__splink__input_table_0,__splink__input_table_1,ALB-CN-1870-1634,ALB-CN-1880-20111,1864.0,1862.0,2,W,W,1,F,F,1,ELIZABETH,ELIZABETH,4,JANA,JANA,0
451985,-0.316599,0.445357,__splink__input_table_0,__splink__input_table_1,ALB-CN-1870-1878,ALB-CN-1880-28385,1809.0,1811.0,2,W,W,1,M,M,1,J,J,4,NACALA,BAD,1
451947,-0.316599,0.445357,__splink__input_table_0,__splink__input_table_1,ALB-CN-1870-16546,ALB-CN-1880-10100,1858.0,1856.0,2,B,B,1,F,F,1,SARAH,SARAH,4,GRAHAN,TAYLAR,1
451958,-0.316599,0.445357,__splink__input_table_0,__splink__input_table_1,ALB-CN-1870-20438,ALB-CN-1880-1795,1860.0,1858.0,2,B,B,1,F,F,1,FANNIE,FANNIE,4,BRAN,RACARDSAN,1
451957,-0.316599,0.445357,__splink__input_table_0,__splink__input_table_1,ALB-CN-1870-15807,ALB-CN-1880-27157,1866.0,1868.0,2,B,B,1,F,F,1,SUSAN,SUSAN,4,CAPAR,HASTAN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2653,-0.970716,0.337859,__splink__input_table_0,__splink__input_table_1,ALB-CN-1870-2105,ALB-CN-1880-13324,1867.0,1845.0,0,W,B,0,M,M,1,JAMES,JAMES,4,PAGA,PAGA,0
2652,-0.970716,0.337859,__splink__input_table_0,__splink__input_table_1,ALB-CN-1870-7171,ALB-CN-1880-1332,1830.0,1843.0,0,B,W,0,M,M,1,JOHN,JOHN,4,WALSAN,WALSAN,0
6285,-0.970716,0.337859,__splink__input_table_0,__splink__input_table_1,ALB-CN-1870-16697,ALB-CN-1880-21788,1840.0,1862.0,0,B,W,0,F,F,1,ELIZABETH,ELIZABETH,4,JANA,JANA,0
6280,-0.970716,0.337859,__splink__input_table_0,__splink__input_table_1,ALB-CN-1870-7865,ALB-CN-1880-24838,1867.0,1855.0,0,B,W,0,M,M,1,WILLIAM,WILLIAM,4,WAD,WAD,0


In [64]:
# looking at john smiths 
predictions = df_predictions.as_pandas_dataframe()


In [ ]:
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    df_predictions, threshold_match_probability=0.5 # CAN PLAY WITH THE THRESHOLD
)
clusters.as_pandas_dataframe(limit=10)

Completed iteration 1, num edges remaining to process: 0


,cluster_id,unique_id,source_year,county,middle_name,birth_year,gender,norm_first_name,nysiis_last_name,soundex_last_name,norm_race,norm_occupation,head,household_id,family_id,narrative,source_dataset
0,__splink__input_table_0-__-ALB-CN-1870-19784,ALB-CN-1870-19784,1870,ALB,R,1865.0,F,CATHERINE,WAYLAD,W453,W,None,False,HC1870-1761,FC1870-3783,Kate R Wayland (F / W born 1865 in Alb). In ho...,__splink__input_table_0
1,__splink__input_table_0-__-ALB-CN-1870-19785,ALB-CN-1870-19785,1870,ALB,B,1867.0,F,SUSAN,WAYLAD,W453,W,None,False,HC1870-1761,FC1870-3783,Susan B Wayland (F / W born 1867 in Alb). In h...,__splink__input_table_0
2,__splink__input_table_0-__-ALB-CN-1870-19786,ALB-CN-1870-19786,1870,ALB,None,1838.0,F,ANN,SWAT,S300,B,None,True,HC1870-1761,FC1870-3784,Anna Sweet (F / B born 1838 in Alb). In house ...,__splink__input_table_0
3,__splink__input_table_0-__-ALB-CN-1870-19787,ALB-CN-1870-19787,1870,ALB,F,1867.0,F,JANE,SWAT,S300,B,None,False,HC1870-1761,FC1870-3784,Jane F Sweet (F / B born 1867 in Alb). In hous...,__splink__input_table_0
4,__splink__input_table_0-__-ALB-CN-1870-19788,ALB-CN-1870-19788,1870,ALB,None,1865.0,M,EDWARD,SWAT,S300,B,None,False,HC1870-1761,FC1870-3784,Edward Sweet (M / B born 1865 in Alb). In hous...,__splink__input_table_0
5,__splink__input_table_0-__-ALB-CN-1870-19789,ALB-CN-1870-19789,1870,ALB,M,1827.0,M,A,WAD,W320,W,AGRICULTURE,False,HC1870-1762,FC1870-3784,A M Woods (M / W born 1827 in Alb). In house w...,__splink__input_table_0
6,__splink__input_table_0-__-ALB-CN-1870-20765,ALB-CN-1870-20765,1870,ALB,None,1866.0,M,JAMES,PA,P000,B,None,False,HC1870-1945,FC1870-3967,James Paw (M / B born 1866 in Alb). In house w...,__splink__input_table_0
7,__splink__input_table_0-__-ALB-CN-1870-20354,ALB-CN-1870-20354,1870,ALB,None,1855.0,F,CAROLINE,WAD,W300,W,DOMESTIC,False,HC1870-1865,FC1870-3886,Caroline Wood (F / W born 1855 in Alb). Occupa...,__splink__input_table_0
8,__splink__input_table_0-__-ALB-CN-1870-20766,ALB-CN-1870-20766,1870,ALB,None,1868.0,M,EDWARD,PA,P000,B,None,False,HC1870-1945,FC1870-3967,Edward Paw (M / B born 1868 in Alb). In house ...,__splink__input_table_0
9,__splink__input_table_0-__-ALB-CN-1870-8829,ALB-CN-1870-8829,1870,ALB,None,1853.0,F,JENNIE,LANA,L500,B,None,False,HC1870-1585,FC1870-1858,Jennie Lane (F / B born 1853 in Alb). In house...,__splink__input_table_0


In [66]:
records_to_view = df_predictions.as_record_dict(limit=100)
linker.visualisations.waterfall_chart(records_to_view, filter_nulls=False)

ValueError: retain_intermediate_calculation_columns and retain_matching_columns must both be set to True in your settings dictionary to use this function, because otherwise the necessary columns will not be available in the input records. Their current values are False and True, respectively. Please re-run your linkage with them both set to True.

### Match Weights Waterfall Chart

The above figure allows you to look at a specific record match and visualizes how each variable player a role in the final match score. 

- look at 11



### Some Key Takeaways

- edit the blocking again 
    - blocking only on race and gender does alot of unneccessary comparisons (simply look at the visualization above)
- edit the deterministic rule when estimating the u and m probabilities
    - its so low that none of the final scores are even positive 
    - i could've been too restrictive

In [ ]:
linker.visualisations.comparison_viewer_dashboard(df_predictions, "scv.html", overwrite=True)

# You can view the scv.html file in your browser, or inline in a notbook as follows
from IPython.display import IFrame

IFrame(src="scv.html", width="100%", height=1200)